# Digital Twin Augsburg – Georgsvorstadt
## Explorative Datenanalyse

Dieses Notebook visualisiert die Gebäudedaten des Quartiers und gibt einen Überblick über
Höhenverteilung, Datenquellen (CityGML / OSM / Fallback), Nutzungsklassen und Datenvollständigkeit.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

from configs.settings import BUILDINGS_GEOJSON, DATA_INTERIM

In [ ]:
# Bereinigtes GeoJSON laden (nach Phase 2)
clean_path = DATA_INTERIM / 'georgsvorstadt_clean.geojson'
gdf = gpd.read_file(clean_path)
gdf_metric = gdf.to_crs('EPSG:25832')

print(f'Gebäude gesamt   : {len(gdf)}')
print(f'CRS              : {gdf.crs}')
print(f'Spalten          : {list(gdf.columns)}')
print()
# Höhenquellen-Übersicht
if 'height_source' in gdf.columns:
    print('Höhenquellen:')
    print(gdf['height_source'].value_counts().to_string())
gdf.head(3)

## 1 · Karte der Gebäudegrundrisse

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

gdf_metric.plot(
    ax=ax,
    column='height_m',
    cmap='YlOrRd',
    legend=True,
    legend_kwds={'label': 'Höhe [m]', 'shrink': 0.6},
    edgecolor='grey',
    linewidth=0.3,
)

ax.set_title('Georgsvorstadt – Gebäudegrundrisse (Farbe = Höhe)', fontsize=14)
ax.set_xlabel('Rechtswert (m)')
ax.set_ylabel('Hochwert (m)')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(DATA_INTERIM.parent / 'output' / 'map_heights.png', dpi=150)
plt.show()

## 2 · Höhenverteilung & Datenquellen

In [ ]:
DEFAULT_H = 9.6  # 3 Geschosse × 3.2 m (Fallback)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogramm der Gebäudehöhen
gdf['height_m'].plot.hist(ax=axes[0], bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(DEFAULT_H, color='red', linestyle='--', label=f'Fallback {DEFAULT_H} m')
axes[0].set_xlabel('Höhe [m]')
axes[0].set_ylabel('Anzahl Gebäude')
axes[0].set_title('Höhenverteilung')
axes[0].legend()

# Höhenquelle aus height_source-Spalte (neue Pipeline)
if 'height_source' in gdf.columns:
    src_counts = gdf['height_source'].value_counts()
    label_map = {'citygml': 'BayernAtlas LoD2', 'osm_tag': 'OSM-Tag', 'default': 'Fallback (kein Tag)'}
    labels = [label_map.get(k, k) for k in src_counts.index]
    colors = ['#2ecc71', '#f39c12', '#e74c3c']
    axes[1].pie(src_counts.values, labels=labels, colors=colors[:len(src_counts)],
                autopct='%1.0f%%', startangle=140)
    axes[1].set_title('Höhenquelle')
    print('Höhenquellen:')
    for k, v in src_counts.items():
        print(f'  {label_map.get(k, k):<25}: {v:>5}  ({v/len(gdf)*100:.0f}%)')
else:
    # Fallback für altes GeoJSON ohne height_source
    has_height = gdf['height'].notna() & (gdf['height'].astype(str).str.strip() != '')
    has_levels = gdf['levels'].notna() & (gdf['levels'].astype(str).str.strip() != '')
    fallback   = ~has_height & ~has_levels
    values = [has_height.sum(), (~has_height & has_levels).sum(), fallback.sum()]
    labels = ['height-Tag', 'levels-Tag', 'Fallback (kein Tag)']
    axes[1].pie(values, labels=labels, colors=['#2ecc71','#f39c12','#e74c3c'],
                autopct='%1.0f%%', startangle=140)
    axes[1].set_title('Höhenquelle (alt)')

plt.tight_layout()
plt.show()

## 3 · CityGML LoD2-Abdeckung

In [ ]:
# Karte: LoD2-Gebäude (grün) vs. OSM-Fallback (rot)
if 'height_source' in gdf.columns:
    gdf_metric['lod2'] = gdf['height_source'] == 'citygml'

    fig, ax = plt.subplots(figsize=(12, 10))
    gdf_metric[gdf_metric['lod2']].plot(
        ax=ax, color='#2ecc71', edgecolor='grey', linewidth=0.2, label='LoD2 (BayernAtlas)'
    )
    gdf_metric[~gdf_metric['lod2']].plot(
        ax=ax, color='#e74c3c', edgecolor='grey', linewidth=0.2, label='OSM / Fallback'
    )

    p1 = mpatches.Patch(color='#2ecc71', label=f"LoD2 BayernAtlas ({gdf_metric['lod2'].sum()})")
    p2 = mpatches.Patch(color='#e74c3c', label=f"OSM/Fallback ({(~gdf_metric['lod2']).sum()})")
    ax.legend(handles=[p1, p2], loc='upper left')
    ax.set_title('CityGML LoD2-Abdeckung', fontsize=14)
    ax.set_xlabel('Rechtswert (m)')
    ax.set_ylabel('Hochwert (m)')
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.savefig(DATA_INTERIM.parent / 'output' / 'map_lod2_coverage.png', dpi=150)
    plt.show()
else:
    print('Spalte height_source fehlt – Phase 2 mit CityGML-Daten erneut ausführen.')

## 4 · Nutzungsklassen

In [ ]:
building_types = gdf['building'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(10, 4))
building_types.plot.bar(ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('OSM building-Tag')
ax.set_ylabel('Anzahl')
ax.set_title('Gebäudenutzung (OSM building-Tag)')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 5 · Grundflächen-Statistik (EPSG:25832)

In [ ]:
gdf_metric['footprint_m2'] = gdf_metric.geometry.area
gdf_metric['levels_num']   = pd.to_numeric(gdf['levels'], errors='coerce').fillna(3)
gdf_metric['gfa_m2']       = gdf_metric['footprint_m2'] * gdf_metric['levels_num']

print('Grundfläche (Footprint):')
print(gdf_metric['footprint_m2'].describe().round(1).to_string())
print(f'\nGesamte Bruttogeschossfläche (GFA): {gdf_metric["gfa_m2"].sum():,.0f} m²')

## 6 · Dachart-Verteilung (CityGML)

In [ ]:
ROOF_CODES = {
    '1000': 'Flachdach', '2100': 'Satteldach', '2200': 'Walmdach',
    '2300': 'Krüppelwalmdach', '2400': 'Mansarddach', '3100': 'Pultdach',
    '3200': 'Vers. Pultdach', '4000': 'Zeltdach', '5000': 'Kegeldach',
    '6000': 'Tonnendach', '9999': 'Sonstiges',
}

if 'roof_type' in gdf.columns:
    # Numerische Codes in Klartextnamen übersetzen (falls noch alte Daten vorliegen)
    roof_col = gdf['roof_type'].fillna('unbekannt').astype(str)
    roof_col = roof_col.map(lambda v: ROOF_CODES.get(v.strip(), v) if v.strip() else 'unbekannt')
    roof = roof_col.value_counts().head(10)

    fig, ax = plt.subplots(figsize=(10, 4))
    roof.plot.bar(ax=ax, color='coral', edgecolor='white')
    ax.set_xlabel('Dachart')
    ax.set_ylabel('Anzahl')
    ax.set_title('Dachart-Verteilung (BayernAtlas LoD2)')
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('Spalte roof_type fehlt – Phase 2 mit CityGML-Daten erneut ausführen.')